# Block 3: Anomaly Detection with Autoencoders
## Hochfrequenz AI Workshop

**Objective:** Detect anomalies in smart meter consumption data (fraud, equipment failure, unusual patterns)

**Key Concepts:** ANN building blocks, Encoder-Decoder architecture, Reconstruction error, Unsupervised learning

**Approach:** Train autoencoder on normal patterns → Detect anomalies via high reconstruction error

## Theory: Autoencoders for Anomaly Detection

### The Anomaly Detection Challenge

**Business Problem (Hochfrequenz Context):**
- Smart meters record millions of consumption patterns
- **Fraud:** Meter tampering, illegal connections (5-10% revenue loss in some markets)
- **Equipment Failure:** Malfunctioning meters, detector issues
- **Unusual Patterns:** Vacant homes with consumption, seasonal anomalies

**Traditional Approach:** Rule-based thresholds
- Example: "Flag if consumption > 2x average"
- **Problem:** Misses complex patterns, high false positives

**ML Approach:** Learn what "normal" looks like, flag deviations

---

### Artificial Neural Networks (ANN) - Building Blocks

Before understanding autoencoders, let's review ANN basics:

**Components:**
1. **Neurons:** Basic units that compute weighted sum + activation
2. **Layers:** Input → Hidden (feature learning) → Output
3. **Activation Functions:** ReLU, sigmoid, tanh (introduce non-linearity)
4. **Backpropagation:** How the network learns (gradient descent)

**Standard ANN Task:** Classification or Regression (supervised)
- Input: Features (consumption, hour, temperature)
- Output: Prediction (next hour consumption, customer type)

---

### Autoencoder Architecture

**Key Difference:** Autoencoder is **unsupervised**
- Input = Output (reconstruct the input itself)
- No labels needed ("normal" vs "anomaly")

**Architecture: Encoder-Decoder**

```
INPUT (24 hours)  →  ENCODER  →  LATENT (compressed)  →  DECODER  →  OUTPUT (24 hours)
     [500W...]          ↓             [4 dims]              ↓           [510W...]
                  Compression                          Reconstruction
```

**Encoder:**
- Compresses input into low-dimensional representation (latent space)
- Example: 24 hours → 4 numbers
- Learns essential features ("morning peak", "weekend pattern")

**Latent Space:**
- Compressed representation capturing key patterns
- Forces network to learn what's important

**Decoder:**
- Reconstructs original input from latent representation
- Example: 4 numbers → 24 hours
- Tries to recover original pattern

---

### How Anomaly Detection Works

**Training Phase:**
1. Train autoencoder on **normal consumption data only**
2. Network learns to reconstruct typical patterns well
3. Loss = Reconstruction Error (how different is output from input)

**Detection Phase:**
1. Feed new consumption pattern (normal or anomalous)
2. Calculate reconstruction error
3. **If error is high → anomaly** (network can't reconstruct it well)
4. **If error is low → normal** (network recognizes pattern)

**Intuition:**
- Autoencoder is like a "pattern expert" trained on normal behavior
- When it sees something unusual, it struggles to reproduce it
- High reconstruction error = "I don't recognize this pattern!"

---

### Why This Works for Energy

**Normal Consumption Patterns:**
- Daily cycle (morning/evening peaks)
- Weekly cycle (weekday vs weekend)
- Seasonal variation (summer/winter)

**Anomalies:**
- Sudden drop to zero (meter failure)
- 30% reduction (tampering/fraud)
- Flat consumption (detector stuck)
- Irregular spikes (unauthorized connection)

Autoencoder learns normal patterns → flags anything that doesn't fit!

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"✅ All libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")

### 1.1 Load Consumption Data

In [ ]:
# Generate realistic synthetic German household consumption data
np.random.seed(42)

# Generate 2 years of hourly data
date_range = pd.date_range(start='2018-05-01', end='2020-12-31', freq='H')
n_samples = len(date_range)

# Create realistic consumption pattern
data = pd.DataFrame({
    'timestamp': date_range,
    'hour': date_range.hour,
    'day_of_week': date_range.dayofweek,
    'month': date_range.month,
    'is_weekend': (date_range.dayofweek >= 5).astype(int)
})

# Hourly pattern
hourly_pattern = np.array([
    300, 280, 250, 240, 250, 350, 550, 700,  # Night to morning
    600, 500, 450, 420, 430, 480, 550, 600,  # Midday
    700, 800, 750, 700, 650, 600, 550, 400   # Evening to night
])

data['consumption_base'] = hourly_pattern[data['hour']]

# Seasonal variation
seasonal_factor = 1.0 + 0.3 * np.sin(2 * np.pi * (data['month'] - 2) / 12)
data['consumption_seasonal'] = data['consumption_base'] * seasonal_factor

# Weekend effect
weekend_factor = data['is_weekend'].apply(lambda x: 0.85 if x else 1.0)
data['consumption_adjusted'] = data['consumption_seasonal'] * weekend_factor

# Add noise
data['consumption'] = data['consumption_adjusted'] + np.random.normal(0, 50, n_samples)
data['consumption'] = data['consumption'].clip(lower=50)

data.set_index('timestamp', inplace=True)

print(f"Dataset shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")
print(f"\nConsumption statistics:")
print(data['consumption'].describe())

## 2. Prepare Data for Autoencoder

**Key Difference from Blocks 1-2:**
- We create rolling windows of 24 hours
- Each window is treated as one "sample"
- Autoencoder learns to reconstruct these 24-hour patterns

In [ ]:
def create_windows(data, window_size=24):
    """
    Create rolling windows of consumption data.
    
    Parameters:
    -----------
    data : array
        Time series data (1D)
    window_size : int
        Size of rolling window (default: 24 hours)
    
    Returns:
    --------
    windows : numpy array (samples, window_size)
        Rolling windows
    """
    windows = []
    for i in range(len(data) - window_size + 1):
        windows.append(data[i:i+window_size])
    return np.array(windows)

# Create 24-hour rolling windows
window_size = 24
consumption_values = data['consumption'].values
X_windows = create_windows(consumption_values, window_size=window_size)

print(f"Rolling windows created:")
print(f"  Shape: {X_windows.shape} (samples, hours)")
print(f"  Each sample = {window_size} consecutive hours of consumption")
print(f"\nExample window (first 24 hours):")
print(X_windows[0])

### 2.1 Normalize Data

In [ ]:
# Normalize windows (important for neural networks)
scaler = StandardScaler()
X_windows_scaled = scaler.fit_transform(X_windows)

print(f"Scaled data:")
print(f"  Mean: {X_windows_scaled.mean():.4f} (should be ~0)")
print(f"  Std:  {X_windows_scaled.std():.4f} (should be ~1)")
print(f"\nExample scaled window:")
print(X_windows_scaled[0])

### 2.2 Train-Test Split

**Important:** We use only "normal" data for training
- First 80%: Training (assume all normal)
- Last 20%: Testing (may contain anomalies)

In [ ]:
# Chronological split
split_idx = int(len(X_windows_scaled) * 0.8)

X_train_normal = X_windows_scaled[:split_idx]
X_test = X_windows_scaled[split_idx:]

print(f"Train set (normal data): {X_train_normal.shape[0]:,} samples")
print(f"Test set:                 {X_test.shape[0]:,} samples")
print(f"\nTrain set is 100% normal consumption (by assumption)")
print(f"Test set may contain anomalies (we'll inject some later)")

## 3. Build Autoencoder Model

**Architecture:**
- Input: 24 hours of consumption
- Encoder: 24 → 16 → 8 → 4 (compression)
- Latent: 4-dimensional representation
- Decoder: 4 → 8 → 16 → 24 (reconstruction)
- Output: 24 hours reconstructed

**Compression Ratio:** 24 → 4 = 6:1 (forces learning of key patterns)

In [ ]:
# Input dimension
input_dim = window_size  # 24
latent_dim = 4  # Compressed representation

# ENCODER: Compress input
encoder = Sequential([
    layers.Dense(16, activation='relu', input_dim=input_dim, name='encoder_1'),
    layers.Dense(8, activation='relu', name='encoder_2'),
    layers.Dense(latent_dim, activation='relu', name='latent')  # Bottleneck
], name='Encoder')

# DECODER: Reconstruct from latent
decoder = Sequential([
    layers.Dense(8, activation='relu', input_dim=latent_dim, name='decoder_1'),
    layers.Dense(16, activation='relu', name='decoder_2'),
    layers.Dense(input_dim, activation='linear', name='output')  # Reconstruct 24 hours
], name='Decoder')

# AUTOENCODER: Encoder + Decoder
autoencoder = Sequential([encoder, decoder], name='Autoencoder')

# Compile
autoencoder.compile(
    optimizer='adam',
    loss='mse',  # Reconstruction error
    metrics=['mae']
)

print(autoencoder.summary())
print(f"\nCompression:")
print(f"  Input: {input_dim} dimensions")
print(f"  Latent: {latent_dim} dimensions")
print(f"  Compression ratio: {latent_dim/input_dim:.1%}")
print(f"\nTotal parameters: {autoencoder.count_params():,}")

## 4. Train Autoencoder on Normal Data

**Key:** Input = Target (unsupervised learning)

In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train (X_train = input AND target)
print("Training autoencoder on normal consumption patterns...\n")
history = autoencoder.fit(
    X_train_normal, X_train_normal,  # Input = Output (unsupervised)
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print("\n✅ Training complete!")

### 4.1 Visualize Training History

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss (MSE)
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Autoencoder Training: Reconstruction Error')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Autoencoder Training: MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_loss = history.history['val_loss'][-1]
print(f"\nFinal validation reconstruction error: {final_loss:.6f}")

### 4.2 Visualize Encoder Latent Space

Let's see what the encoder learned (4D latent representation)

In [ ]:
# Encode training data to latent space
latent_train = encoder.predict(X_train_normal[:1000], verbose=0)  # First 1000 samples

# Plot first 2 dimensions
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(latent_train[:, 0], latent_train[:, 1], 
                     c=range(len(latent_train)), cmap='viridis', alpha=0.6, s=30)
ax.set_xlabel('Latent Dimension 1')
ax.set_ylabel('Latent Dimension 2')
ax.set_title('Autoencoder Latent Space (2D projection)')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Time Order')
plt.tight_layout()
plt.show()

print(f"\nLatent space captures compressed patterns:")
print(f"  4 dimensions encode essential consumption characteristics")
print(f"  Similar patterns cluster together in latent space")

## 5. Anomaly Detection: Calculate Reconstruction Error

**Method:**
1. Reconstruct test data using autoencoder
2. Calculate error (MSE between actual and reconstructed)
3. Set threshold (e.g., 95th percentile)
4. Flag samples above threshold as anomalies

In [ ]:
# Reconstruct test data
X_test_reconstructed = autoencoder.predict(X_test, verbose=0)

# Calculate reconstruction error for each window
reconstruction_error = np.mean(np.power(X_test - X_test_reconstructed, 2), axis=1)

# Set threshold (95th percentile = top 5% are anomalies)
threshold_percentile = 95
threshold = np.percentile(reconstruction_error, threshold_percentile)

# Detect anomalies
anomalies = reconstruction_error > threshold
anomaly_indices = np.where(anomalies)[0]

print(f"="*60)
print(f"ANOMALY DETECTION RESULTS")
print(f"="*60)
print(f"Reconstruction Error Statistics:")
print(f"  Mean:   {reconstruction_error.mean():.6f}")
print(f"  Median: {np.median(reconstruction_error):.6f}")
print(f"  Std:    {reconstruction_error.std():.6f}")
print(f"  Min:    {reconstruction_error.min():.6f}")
print(f"  Max:    {reconstruction_error.max():.6f}")
print(f"\nThreshold (95th percentile): {threshold:.6f}")
print(f"\nAnomalies detected: {anomalies.sum()} / {len(reconstruction_error)} ({anomalies.sum()/len(reconstruction_error)*100:.1f}%)")
print(f"="*60)

## 6. Visualization: Anomaly Detection Results

In [ ]:
# Plot reconstruction error time series
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# TOP: Reconstruction error over time
axes[0].plot(reconstruction_error, linewidth=1.5, alpha=0.7, color='steelblue')
axes[0].axhline(y=threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold ({threshold_percentile}th percentile): {threshold:.4f}')
axes[0].scatter(anomaly_indices, reconstruction_error[anomaly_indices], 
                color='red', s=80, marker='x', linewidths=2, label='Anomalies', zorder=5)
axes[0].set_ylabel('Reconstruction Error (MSE)')
axes[0].set_title('Anomaly Detection: Reconstruction Error Over Time')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# BOTTOM: Compare normal vs anomalous pattern reconstruction
# Find best and worst reconstructions
normal_idx = np.argmin(reconstruction_error)  # Best reconstruction (most normal)
anomaly_idx = anomaly_indices[0] if len(anomaly_indices) > 0 else np.argmax(reconstruction_error)  # Worst reconstruction

hours = range(24)
axes[1].plot(hours, X_test[normal_idx], 'o-', label='Normal (actual)', 
             linewidth=2.5, markersize=6, color='green')
axes[1].plot(hours, X_test_reconstructed[normal_idx], 's--', label='Normal (reconstructed)', 
             linewidth=2.5, markersize=6, color='lightgreen')

axes[1].plot(hours, X_test[anomaly_idx], 'o-', label='Anomaly (actual)', 
             linewidth=2.5, markersize=6, color='red')
axes[1].plot(hours, X_test_reconstructed[anomaly_idx], 's--', label='Anomaly (reconstructed)', 
             linewidth=2.5, markersize=6, color='pink')

axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Consumption (normalized)')
axes[1].set_title('Reconstruction Comparison: Normal vs Anomalous Pattern')
axes[1].set_xticks(range(0, 24, 3))
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Add error annotations
axes[1].text(0.02, 0.95, f'Normal error: {reconstruction_error[normal_idx]:.4f}', 
             transform=axes[1].transAxes, va='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
axes[1].text(0.02, 0.85, f'Anomaly error: {reconstruction_error[anomaly_idx]:.4f}', 
             transform=axes[1].transAxes, va='top',
             bbox=dict(boxstyle='round', facecolor='pink', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\n📊 Visualization Insights:")
print(f"  - Normal patterns: Low reconstruction error (autoencoder recognizes pattern)")
print(f"  - Anomalous patterns: High reconstruction error (autoencoder struggles)")

## 7. Case Study: Synthetic Fraud Detection

Let's test if the autoencoder can detect meter tampering (consumption reduction)

In [ ]:
# Create synthetic fraud dataset
X_test_with_fraud = X_test.copy()

# Inject fraud: randomly reduce consumption by 30-50% in 5% of test windows
np.random.seed(123)
fraud_rate = 0.05
n_fraud = int(len(X_test) * fraud_rate)
fraud_indices = np.random.choice(len(X_test), size=n_fraud, replace=False)

for idx in fraud_indices:
    reduction_factor = np.random.uniform(0.5, 0.7)  # 30-50% reduction
    X_test_with_fraud[idx] *= reduction_factor

# Create ground truth labels
y_true = np.zeros(len(X_test), dtype=int)
y_true[fraud_indices] = 1  # 1 = fraud

print(f"Synthetic fraud dataset created:")
print(f"  Total samples: {len(X_test)}")
print(f"  Fraud injected: {n_fraud} samples ({fraud_rate*100:.0f}%)")
print(f"  Normal: {len(X_test) - n_fraud} samples ({(1-fraud_rate)*100:.0f}%)")

### 7.1 Detect Fraud with Autoencoder

In [ ]:
# Reconstruct fraud dataset
X_fraud_reconstructed = autoencoder.predict(X_test_with_fraud, verbose=0)

# Calculate reconstruction error
reconstruction_error_fraud = np.mean(np.power(X_test_with_fraud - X_fraud_reconstructed, 2), axis=1)

# Detect anomalies (use same threshold)
fraud_detected = reconstruction_error_fraud > threshold

# Evaluate detection performance
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

precision = precision_score(y_true, fraud_detected)
recall = recall_score(y_true, fraud_detected)
f1 = f1_score(y_true, fraud_detected)
accuracy = accuracy_score(y_true, fraud_detected)

print(f"="*60)
print(f"FRAUD DETECTION PERFORMANCE")
print(f"="*60)
print(f"Ground Truth:")
print(f"  Fraud cases: {y_true.sum()}")
print(f"  Normal cases: {len(y_true) - y_true.sum()}")
print(f"\nDetection Results:")
print(f"  Flagged as anomaly: {fraud_detected.sum()}")
print(f"  True positives (fraud caught): {np.sum((y_true == 1) & (fraud_detected == 1))}")
print(f"  False positives (normal flagged): {np.sum((y_true == 0) & (fraud_detected == 1))}")
print(f"  False negatives (fraud missed): {np.sum((y_true == 1) & (fraud_detected == 0))}")
print(f"\nMetrics:")
print(f"  Accuracy:  {accuracy:.3f}")
print(f"  Precision: {precision:.3f} (of flagged cases, % truly fraud)")
print(f"  Recall:    {recall:.3f} (of fraud cases, % detected)")
print(f"  F1-Score:  {f1:.3f}")
print(f"="*60)

### 7.2 Confusion Matrix

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(y_true, fraud_detected)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Fraud'],
            ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title('Fraud Detection Confusion Matrix')
plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"  Top-left: True negatives (correctly identified normal)")
print(f"  Bottom-right: True positives (correctly detected fraud)")
print(f"  Top-right: False positives (normal flagged as fraud)")
print(f"  Bottom-left: False negatives (fraud missed)")

### 7.3 Visualize Fraud Examples

In [ ]:
# Plot examples of detected fraud
detected_fraud_indices = np.where((y_true == 1) & (fraud_detected == 1))[0]

if len(detected_fraud_indices) >= 2:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for i, idx in enumerate(detected_fraud_indices[:4]):
        row = i // 2
        col = i % 2
        
        hours = range(24)
        axes[row, col].plot(hours, X_test[idx], 'o-', label='Original (normal)', 
                           linewidth=2, markersize=6, color='green')
        axes[row, col].plot(hours, X_test_with_fraud[idx], 's-', label='With fraud (reduced)', 
                           linewidth=2, markersize=6, color='red')
        axes[row, col].plot(hours, X_fraud_reconstructed[idx], 'x--', label='Reconstructed', 
                           linewidth=2, markersize=6, color='orange', alpha=0.7)
        
        axes[row, col].set_xlabel('Hour')
        axes[row, col].set_ylabel('Consumption (normalized)')
        axes[row, col].set_title(f'Detected Fraud Example {i+1} (Error: {reconstruction_error_fraud[idx]:.4f})')
        axes[row, col].set_xticks(range(0, 24, 4))
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Successfully detected {len(detected_fraud_indices)} fraud cases!")
    print(f"   Autoencoder recognizes that reduced consumption doesn't match learned patterns")
else:
    print(f"\n⚠️ Few fraud cases detected. Try adjusting threshold or fraud injection parameters.")

## 8. Real-World Applications for Hochfrequenz Clients

### Use Case 1: Meter Fraud Detection
- **Problem:** 5-10% revenue loss from meter tampering
- **Solution:** Autoencoder flags unusual consumption drops
- **Impact:** Prioritize field inspections, reduce fraud losses

### Use Case 2: Equipment Failure Detection
- **Problem:** Faulty meters report incorrect data
- **Solution:** Detect flat/stuck consumption patterns
- **Impact:** Proactive maintenance, better data quality

### Use Case 3: Billing Dispute Resolution
- **Problem:** Customers dispute high bills
- **Solution:** Compare consumption to "normal" profile
- **Impact:** Data-driven customer service, faster resolution

### Use Case 4: Grid Monitoring
- **Problem:** Unusual consumption may indicate grid issues
- **Solution:** Real-time anomaly detection across network
- **Impact:** Early warning system, prevent outages

## 9. Key Takeaways

### What We Learned:

1. **Autoencoders are Unsupervised:** No labels needed, just normal data
2. **Encoder-Decoder Structure:** Compression forces learning of key patterns
3. **Reconstruction Error = Anomaly Score:** High error = unusual pattern
4. **Fraud Detection Works:** Achieved ~{:.1f}% recall with {:.1f}% precision
5. **Threshold Selection Matters:** 95th percentile is a good starting point

### Advantages:
- No labeled anomaly data needed
- Learns complex patterns automatically
- Adapts to new normal patterns (retrain periodically)
- Scalable to millions of meters

### Limitations:
- Needs "clean" training data (all normal)
- Threshold selection can be tricky
- May have false positives (legitimate unusual patterns)
- Requires retraining as patterns evolve

### Production Considerations:
- **Retraining:** Monthly or quarterly (capture seasonal changes)
- **Threshold Tuning:** Balance false positives vs false negatives
- **Human Review:** Flagged cases should be investigated, not automatically acted on
- **Explainability:** Show which hours contributed most to anomaly score

---

### Next: Block 4 - Integrated Project
We'll combine forecasting (Blocks 1-2) + anomaly detection (Block 3) + customer segmentation into one complete pipeline!
""".format(recall * 100, precision * 100)